In [ ]:
import numpy as np
import os
import scipy
import pandas as pd
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh
import matplotlib.pyplot as plt

In [ ]:
TongJi_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\TongJi_dataset'
package_list=os.listdir(TongJi_path)
TongJi_data={}
TongJi_SOH={}
for i,package in enumerate(package_list):
    package_name=os.listdir(os.path.join(TongJi_path,package))[0]
    print(f'package: {package_name}')
    battery_list=os.listdir(os.path.join(TongJi_path,package,package_name))

    #print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_path=os.path.join(os.path.join(TongJi_path,package,package_name),battery)
        battery_data=pd.read_csv(battery_path).dropna(subset=[ 'Q discharge/mA.h','Q charge/mA.h'])
        cycle_list= battery_data['cycle number'].unique()
        cycle_list=sorted(cycle_list)

        #print(cycle_list)
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        for cycle in cycle_list:
            #print(cycle)

            voltage=[]
            current=[]
            time=[]
            capacity=[]

            #print(cycle_num)

            cycle_data=battery_data[battery_data['cycle number']==cycle]
            voltage=cycle_data['Ecell/V'].values.reshape(1,-1)

            current=cycle_data['<I>/mA'].values.reshape(1,-1)*0.001
            time_segment = cycle_data['time/s'].values.reshape(1,-1)
            time=time_segment  # 转换为秒
            #time=time.reshape(1,-1)
            discharge_capacity=cycle_data['Q discharge/mA.h'].values.reshape(1,-1)*0.001

            #discharge_capacity=discharge_capacity-discharge_capacity[0][0]
            #print(discharge_capacity)
            charge_capacity=cycle_data['Q charge/mA.h'].values.reshape(1,-1)*0.001
            discharge_capacity=np.float32(discharge_capacity)

            charge_capacity=np.float32(charge_capacity)
            capacity=np.concatenate((discharge_capacity,charge_capacity),axis=1)
            #print(discharge_capacity.shape)
            capacity_max=np.max(capacity)
            if i==2:
                soh=capacity_max/2.5
            else:
                soh=capacity_max/3.3

            #print(soh)
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)
    TongJi_data[f'package_{i+1}']=package1
    TongJi_SOH[f'package_{i+1}']=package2

In [ ]:
battery_soh_plot(TongJi_SOH,TongJi_SOH['package_1'].keys(),package='package_1')

In [ ]:
smooth_TongJi_SOH=smooth_soh(TongJi_SOH,method='moving_average', sigma=4)
package='package_1'
battery_soh_plot(smooth_TongJi_SOH,smooth_TongJi_SOH[package].keys(),package)

In [ ]:
fig_plot(TongJi_SOH['package_1']['battery_4'])

In [ ]:
for package in TongJi_data.keys():
    for battery in TongJi_data[package].keys():
        if len(TongJi_data[package][battery])!= len(TongJi_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(TongJi_data[package][battery])}, SOH length: {len(TongJi_SOH[package][battery])}')


In [ ]:
for package in TongJi_data.keys():
    for battery in TongJi_data[package].keys():
        TongJi_data[package][battery] = TongJi_data[package][battery][3:][:][:]

In [ ]:
for package in TongJi_data.keys():
    for battery in TongJi_data[package].keys():
        if len(TongJi_data[package][battery])!= len(TongJi_SOH[package][battery]):
            print(f'battery {battery} has different length of data and SOH,data length: {len(TongJi_data[package][battery])}, SOH length: {len(TongJi_SOH[package][battery])}')

In [ ]:
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\TongJi_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(TongJi_data,os.path.join(save_path,'TongJi_data.pkl'))
save_data(TongJi_SOH,os.path.join(save_path,'TongJi_SOH.pkl'))
TongJi_data=load_data(os.path.join(save_path,'TongJi_data.pkl'))
TongJi_SOH=load_data(os.path.join(save_path,'TongJi_SOH.pkl'))


In [ ]:
"""TongJi_NCA_data={}
TongJi_NCA_SOH={}
TongJi_NCA_data['package_1']=TongJi_data['package_1']
TongJi_NCA_SOH['package_1']=TongJi_SOH['package_1']
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\TongJi_NCA_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(TongJi_NCA_data,os.path.join(save_path,'TongJi_NCA_data.pkl'))
save_data(TongJi_NCA_SOH,os.path.join(save_path,'TongJi_NCA_SOH.pkl'))

TongJi_NCM_data={}
TongJi_NCM_SOH={}
TongJi_NCM_data['package_1']=TongJi_data['package_2']
TongJi_NCM_SOH['package_1']=TongJi_SOH['package_2']
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\TongJi_NCM_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(TongJi_NCM_data,os.path.join(save_path,'TongJi_NCM_data.pkl'))
save_data(TongJi_NCM_SOH,os.path.join(save_path,'TongJi_NCM_SOH.pkl'))

TongJi_NCMNCA_data={}
TongJi_NCMNCA_SOH={}
TongJi_NCMNCA_data['package_1']=TongJi_data['package_3']
TongJi_NCMNCA_SOH['package_1']=TongJi_SOH['package_3']
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\TongJi_NCMNCA_dataset'
os.makedirs(save_path,exist_ok=True)
save_data(TongJi_NCMNCA_data,os.path.join(save_path,'TongJi_NCMNCA_data.pkl'))
save_data(TongJi_NCMNCA_SOH,os.path.join(save_path,'TongJi_NCMNCA_SOH.pkl'))"""


In [ ]:
fig_plot(TongJi_data['package_1']['battery_4'][0][2])